# MVA-MECHANORESCUE · NB01C — TARGETED BUB1B PHASE + ALLELE AUDIT
**v1.0.0 · privacy-preserving · VCF/PGT/PID + private VEP only · no external variant API**

## Purpose

NB01B identified a robust top pair in **BUB1B**:
- one `stop_gained` allele,
- one `missense_variant`,
- `rank_no_mva_prior = 1`,
- but `phase_status = phase_unknown`.

This notebook does **not** assume trans phase.

It:
1. reads only the sanitized `NB01B_HANDOFF.json` plus local/private VCFs on Drive;
2. queries **only the two target BUB1B loci**;
3. audits GT/GQ/DP/AD/PGT/PID/PL locally;
4. audits the target interval for phase-block context;
5. queries the private VEP VCF only at the two target loci for transcript/protein/SIFT/PolyPhen/ClinVar/AF evidence;
6. writes a sanitized public handoff containing only target-pair evidence;
7. returns a hard gate:
   - `TRANS_SUPPORTED`,
   - `CIS_SUPPORTED`,
   - or `PHASE_UNRESOLVED`.

No raw VCF rows, full private parquet, sample name, or read-level data are exported.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("cyvcf2") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "cyvcf2"
    ])

from cyvcf2 import VCF
print("cyvcf2 import: PASS")

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone
import json, re
import numpy as np
import pandas as pd
from cyvcf2 import VCF
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
DATA_DIR = ROOT / "00_PRIVATE_DATA" / "hf_snapshot"
PRIVATE_NB01B = ROOT / "02_RESULTS_PRIVATE" / "NB01B_COMPOUND_HET_v1_0_0"
PRIVATE_OUT = ROOT / "02_RESULTS_PRIVATE" / "NB01C_TARGETED_BUB1B_v1_0_0"
PUBLIC_OUT = ROOT / "03_PUBLIC_ARTIFACTS"
HANDOFF = PUBLIC_OUT / "NB01B_HANDOFF.json"

PRIVATE_OUT.mkdir(parents=True, exist_ok=True)
PUBLIC_OUT.mkdir(parents=True, exist_ok=True)

assert HANDOFF.exists(), HANDOFF
h = json.loads(HANDOFF.read_text())
tp = h["top_pair"]

assert tp["gene"] == "BUB1B", f"Expected BUB1B top pair, got {tp['gene']}"
assert tp["phase_status"] == "phase_unknown", tp["phase_status"]

def parse_key(k):
    chrom, pos, ref, alt = k.split(":", 3)
    return {"chrom": chrom, "pos": int(pos), "ref": ref, "alt": alt}

v1 = parse_key(tp["variant1"])
v2 = parse_key(tp["variant2"])

print("NB01B top pair:", tp["gene"])
print("Variant classes:", tp["consequence_1"], "+", tp["consequence_2"])
print("Distance bp:", abs(v2["pos"] - v1["pos"]))
print("No-MVA-prior rank:", tp["rank_no_mva_prior"])

## 1. Input inventory and phase-data availability

In [ ]:
vcfs = sorted(list(DATA_DIR.rglob("*.vcf.gz")) + list(DATA_DIR.rglob("*.vcf")))
bams = sorted(DATA_DIR.rglob("*.bam"))
crams = sorted(DATA_DIR.rglob("*.cram"))
fastqs = sorted(DATA_DIR.rglob("*.fastq.gz")) + sorted(DATA_DIR.rglob("*.fq.gz"))

assert len(vcfs) == 1, f"Expected one proband VCF, found {len(vcfs)}"
VCF_IN = vcfs[0]

raw_read_status = {
    "bam_count": len(bams),
    "cram_count": len(crams),
    "fastq_count": len(fastqs),
    "read_backed_phasing_possible_from_snapshot": bool(bams or crams or fastqs),
}
print(json.dumps(raw_read_status, indent=2))

if not raw_read_status["read_backed_phasing_possible_from_snapshot"]:
    print("NOTE: no BAM/CRAM/FASTQ in the current snapshot. NB01C will not claim read-backed phase.")

## 2. Query only the two target loci from the private VCF

In [ ]:
def scalar_format(var, key):
    try:
        x = var.format(key)
        if x is None or len(x) == 0:
            return None
        x = x[0]
        if isinstance(x, np.ndarray):
            if x.size == 0: return None
            if x.size == 1: x = x.item()
            else: x = x.tolist()
        if isinstance(x, (bytes, bytearray)):
            return x.decode(errors="ignore").rstrip("\x00")
        if isinstance(x, np.generic):
            return x.item()
        return x
    except Exception:
        return None

def alt_hap_from_pgt(pgt, allele_num=1):
    if pgt is None:
        return None
    p = str(pgt).strip()
    if "|" not in p:
        return None
    parts = p.split("|")
    hits = [i for i,a in enumerate(parts) if a == str(allele_num)]
    if len(hits) == 1: return hits[0]
    if len(hits) == 2: return 2
    return None

def query_exact(vcf, q):
    region = f'{q["chrom"]}:{q["pos"]}-{q["pos"]}'
    hits = []
    for var in vcf(region):
        if int(var.POS) != q["pos"] or str(var.REF) != q["ref"]:
            continue
        if q["alt"] not in list(var.ALT):
            continue
        allele_num = list(var.ALT).index(q["alt"]) + 1
        gt = var.genotypes[0] if var.genotypes else None
        hits.append({
            "chrom": str(var.CHROM),
            "pos": int(var.POS),
            "ref": str(var.REF),
            "alt": q["alt"],
            "allele_num": allele_num,
            "gt": None if gt is None else gt[:2],
            "gt_phased_flag": None if gt is None or len(gt) < 3 else bool(gt[2]),
            "gq": scalar_format(var, "GQ"),
            "dp": scalar_format(var, "DP"),
            "ad": scalar_format(var, "AD"),
            "pgt": scalar_format(var, "PGT"),
            "pid": scalar_format(var, "PID"),
            "pl": scalar_format(var, "PL"),
            "alt_haplotype": alt_hap_from_pgt(scalar_format(var, "PGT"), allele_num),
        })
    return hits

v = VCF(str(VCF_IN))
hits1 = query_exact(v, v1)
hits2 = query_exact(v, v2)
v.close()

assert len(hits1) == 1, f"Variant1 exact hits: {len(hits1)}"
assert len(hits2) == 1, f"Variant2 exact hits: {len(hits2)}"
a, b = hits1[0], hits2[0]

target_private = pd.DataFrame([a,b])
target_private.to_json(PRIVATE_OUT/"target_genotypes_private.json", orient="records", indent=2)

def public_gt(x):
    return {
        "variant": f'{x["chrom"]}:{x["pos"]}:{x["ref"]}:{x["alt"]}',
        "gt": x["gt"],
        "gt_phased_flag": x["gt_phased_flag"],
        "gq": x["gq"],
        "dp": x["dp"],
        "pgt": x["pgt"],
        "pid_present": x["pid"] is not None,
        "same_pid_value_disclosed": False,
        "alt_haplotype": x["alt_haplotype"],
    }

print(json.dumps({"variant1": public_gt(a), "variant2": public_gt(b)}, indent=2, default=str))

## 3. Phase gate

In [ ]:
def is_het(x):
    gt = x.get("gt")
    return isinstance(gt, (list, tuple)) and len(gt) >= 2 and sorted(gt) == [0,1]

phase_reason = []
if not (is_het(a) and is_het(b)):
    phase_status = "GENOTYPE_NOT_SIMPLE_HET"
    phase_reason.append("At least one target allele is not a simple 0/1 heterozygous genotype.")
elif a.get("pid") is not None and b.get("pid") is not None and str(a["pid"]) == str(b["pid"]):
    ha, hb = a.get("alt_haplotype"), b.get("alt_haplotype")
    if ha in (0,1) and hb in (0,1):
        if ha != hb:
            phase_status = "TRANS_SUPPORTED"
            phase_reason.append("Both target variants are in the same PGT/PID phase block on opposite haplotypes.")
        else:
            phase_status = "CIS_SUPPORTED"
            phase_reason.append("Both target variants are in the same PGT/PID phase block on the same haplotype.")
    else:
        phase_status = "PHASE_UNRESOLVED"
        phase_reason.append("Shared PID exists but one/both PGT haplotypes are not interpretable as 0/1.")
else:
    phase_status = "PHASE_UNRESOLVED"
    phase_reason.append("Target variants do not share an interpretable PGT/PID block.")

print("PHASE STATUS:", phase_status)
for x in phase_reason:
    print("-", x)

## 4. Local phase-block context across the BUB1B interval

In [ ]:
start = min(v1["pos"], v2["pos"])
end = max(v1["pos"], v2["pos"])
region = f'{v1["chrom"]}:{start}-{end}'

ctx = []
v = VCF(str(VCF_IN))
for var in v(region):
    gt = var.genotypes[0] if var.genotypes else None
    if gt is None or len(gt) < 2 or sorted(gt[:2]) != [0,1]:
        continue
    pgt = scalar_format(var, "PGT")
    pid = scalar_format(var, "PID")
    if pgt is None and pid is None:
        continue
    ctx.append({
        "pos": int(var.POS),
        "pgt": pgt,
        "pid": pid,
        "gt_phased_flag": None if len(gt)<3 else bool(gt[2]),
    })
v.close()

ctx_df = pd.DataFrame(ctx)
ctx_df.to_json(PRIVATE_OUT/"phase_block_context_private.json", orient="records", indent=2)

summary = {
    "interval_bp": end-start,
    "phased_het_markers_in_interval": int(len(ctx_df)),
    "distinct_pid_blocks": int(ctx_df["pid"].dropna().astype(str).nunique()) if not ctx_df.empty else 0,
    "target_variants_share_pid": bool(
        a.get("pid") is not None and b.get("pid") is not None and str(a["pid"]) == str(b["pid"])
    ),
}
print(json.dumps(summary, indent=2))

## 5. Target-only private VEP audit

In [ ]:
VEP_VCF = PRIVATE_NB01B / "proband_vep116.vcf.gz"
assert VEP_VCF.exists(), VEP_VCF

vv = VCF(str(VEP_VCF))
csq_header = vv.get_header_type("CSQ")["Description"]
m = re.search(r"Format:\s*(.+?)\"?$", csq_header)
assert m, "Cannot parse CSQ header"
fields = m.group(1).rstrip('">').split("|")
idx = {k:i for i,k in enumerate(fields)}

def cg(parts, key):
    i=idx.get(key)
    return "" if i is None or i>=len(parts) else parts[i]

def exact_csq(q):
    region=f'{q["chrom"]}:{q["pos"]}-{q["pos"]}'
    out=[]
    for var in vv(region):
        if int(var.POS)!=q["pos"] or str(var.REF)!=q["ref"] or q["alt"] not in list(var.ALT):
            continue
        anum=list(var.ALT).index(q["alt"])+1
        raw=var.INFO.get("CSQ")
        if not raw: continue
        for block in str(raw).split(","):
            p=block.split("|")
            try:
                xnum=int(cg(p,"ALLELE_NUM"))
            except:
                continue
            if xnum != anum: continue
            if cg(p,"SYMBOL") != "BUB1B": continue
            out.append({
                "consequence": cg(p,"Consequence"),
                "impact": cg(p,"IMPACT"),
                "feature": cg(p,"Feature"),
                "canonical": cg(p,"CANONICAL"),
                "mane_select": cg(p,"MANE_SELECT"),
                "protein_id": cg(p,"ENSP"),
                "protein_position": cg(p,"Protein_position"),
                "amino_acids": cg(p,"Amino_acids"),
                "codons": cg(p,"Codons"),
                "existing_variation": cg(p,"Existing_variation"),
                "clin_sig": cg(p,"CLIN_SIG"),
                "sift": cg(p,"SIFT"),
                "polyphen": cg(p,"PolyPhen"),
                "gnomade_af": cg(p,"gnomADe_AF"),
                "gnomadg_af": cg(p,"gnomADg_AF"),
                "max_af": cg(p,"MAX_AF"),
            })
    return out

e1=exact_csq(v1)
e2=exact_csq(v2)
vv.close()

allele_evidence = {"variant1": e1, "variant2": e2}
(PRIVATE_OUT/"target_vep_evidence_private.json").write_text(json.dumps(allele_evidence,indent=2))
print("Target VEP records:", len(e1), len(e2))
print(json.dumps(allele_evidence, indent=2))

## 6. Decision gate and sanitized handoff

In [ ]:
def best_record(xs):
    if not xs: return {}
    return sorted(xs, key=lambda r: (
        bool(r.get("mane_select")),
        r.get("canonical") == "YES"
    ), reverse=True)[0]

be1, be2 = best_record(e1), best_record(e2)

if phase_status == "TRANS_SUPPORTED":
    next_action = (
        "Proceed to allele-specific BUB1B mechanism notebook: "
        "LoF/NMD/abundance for the stop allele plus local structural/domain/motif analysis for the missense allele."
    )
elif phase_status == "CIS_SUPPORTED":
    next_action = (
        "Do not treat this pair as recessive compound-heterozygous. "
        "Return to NB01B ranking and search for an independent second allele."
    )
else:
    if raw_read_status["read_backed_phasing_possible_from_snapshot"]:
        next_action = "Perform targeted read-backed phasing on the available local read data."
    else:
        next_action = (
            "Phase remains unresolved with the current VCF-only snapshot. "
            "Do not claim compound-heterozygous trans phase. "
            "Request/use BAM/CRAM/FASTQ or parental/family genotype data if available; "
            "meanwhile the BUB1B pair may remain the leading candidate with an explicit phase limitation."
        )

public = {
    "schema": "MVA-NB01C-TARGETED-PHASE-AUDIT-1.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "gene": "BUB1B",
    "variant1": tp["variant1"],
    "variant2": tp["variant2"],
    "nb01b_rank_primary": 1,
    "nb01b_rank_no_mva_prior": tp["rank_no_mva_prior"],
    "phase_status": phase_status,
    "phase_reason": phase_reason,
    "raw_read_data_available_in_snapshot": raw_read_status["read_backed_phasing_possible_from_snapshot"],
    "phase_context": summary,
    "allele1": {
        "nb01b_consequence": tp["consequence_1"],
        "vep": be1,
    },
    "allele2": {
        "nb01b_consequence": tp["consequence_2"],
        "vep": be2,
    },
    "privacy": {
        "external_variant_api_used": False,
        "raw_vcf_exported": False,
        "full_private_variant_table_exported": False,
        "sample_name_exported": False,
        "only_target_pair_evidence_exported": True,
    },
    "next_action": next_action,
}

out = PUBLIC_OUT/"NB01C_BUB1B_PHASE_HANDOFF.json"
out.write_text(json.dumps(public, indent=2))
print(json.dumps(public, indent=2))
print("\nNB01C COMPLETE:", out)

## Interpretation rules

- `TRANS_SUPPORTED` is sufficient to advance the BUB1B pair to allele-specific mechanism work.
- `CIS_SUPPORTED` falsifies this pair as a recessive compound-heterozygous explanation.
- `PHASE_UNRESOLVED` is **not** negative evidence for BUB1B; it is a technical limitation.
- No drug ranking is allowed from NB01C.
- NB03 should be rerun only after this phase gate is reviewed.
- For the mixed `stop_gained + missense` pair, downstream routing must preserve **both allele classes** rather than collapsing the pair to a single `lof` class.